# Genomics on the command line day 3: tabular data

## Setting up the workshop
This workshop is organized as a `jupyter` notebook, which allows us to work in an interactive environment to edit and run code in small "blocks", without requiring you to install various packages yourselves. We'll be running it using [Google Colab](https://colab.research.google.com/): either click on the "Open Jupyter Notebook" link on the [landing page for the workshop](https://informatics.fas.harvard.edu/workshops/biotips/), or if you have downloaded the notebook yourself, just upload it to Colab and open it.

Jupyter notebooks have blocks of formatted text ("text" or "markdown" blocks), as well as "code" blocks that contain executable commands that can be run within the notebook. Clicking on the arrow in the upper left of a code block will execute the code in that block.

### Install software

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

In [ ]:
import condacolab
condacolab.check()

!conda install -c bioconda bedtools

### Download example data

In [ ]:
mkdir -p data_day3/
wget -O data_day3/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/genome.bed
wget -O data_day3/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.bed
wget -O data_day3/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.csv
wget -O data_day3/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/repeats.bed
wget -O data_day3/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/example.sam
wget -O data_day3/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/sv.bed
wget -O data_day3/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/data/genome.fa


mkdir -p img/
wget -O img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/bedtools_getfasta.png
wget -O img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/bedtools_intersect.png
wget -O img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/bedtools_getfasta.png
wget -O img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/bedtools_merge.png
wget -O img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/bedtools_subtract.png
wget -O img/ https://raw.githubusercontent.com/harvardinformatics/biotips/refs/heads/main/img/annotation_coordSystems.png

## Tabular data files
Strictly speaking, "tabular data" does not refer to a specific bioinformatic file format (like a FASTA or a BAM file), but instead refers to a generic type of text file...in fact, you have probably worked with tabular data before without even realizing it! 

"Tabular data" refers to *data that is organized into a table, where ***rows*** (also called "records") are individual observations and ***columns*** (also called "fields") are specific variables*; columns are separated from each other by some specific **delimiter** (or "field separator"), such as a comma, a tab, a blank space, etc. The delimiter is often referred to in the naming of the file, such as:
- `.csv` = "comma-separated values"
- `.tsv` = "tab-separated values"
- `.xlsx` = an Excel spreadsheet is actually just fancy (proprietary) tabular data

Note that unlike previous formats we have looked at, the file extension (i.e. the `.[file type]` that goes at the end of a file name) for tabular data files is not standardized. You can have a tab-separated value file that ends in `.txt`, for example...so as always, it is always a good idea to visually inspect your files before working with them!

The contents of tabular data are also not standardized: all that is required is a consistent number of rows and columns. For example, if we look at a SAM file that we covered in last workshop:

In [ ]:
%%bash
head data_day3/example.sam

We can see that (ignoring the header lines) a SAM file is essentially just a fancy `.tsv` file! And while we would want to use a specialized tool like `samtools` to manipulate SAM files, we could in theory treat it like any other tabular file.

## BED files
Now we are going to introduce a bioinformatics-specific type of tabular data called BED files. Specifically, BED files are a type of *annotation* or *interval* file: each line indicates the coordinates of some defined region in a genome. The BED format is an extremely flexible format -- the regions contained within it can represent anything, such as:

- Annotated genes
- Repetitive regions
- Transcription factor binding sites
- CRISPR cut sites
- Etc.  

In its most basic and common form it is also an extremely *simple* format, consisting of three columns of text separated by a tab character:

- First column represents the reference sequence name 
- Second indicates the starting coordinate 
- Third indicates the ending coordinate

These are the only 3 required columns for a BED file, but they can optionally have additional columns containing any other type of information, such as the type of annotation, the strand, score, etc.

BED files might have the `.bed` extension, and while it is best practice to use a file extension that properly describes the format of a file it is not required. Any 3 column tab delimited file that has the columns we described can be treated as a **BED** file.

#### Coordinate system
Technically speaking, BED files use 0-start half-open coordinate system (aka a "right-open interval"); what this means is that the first base of a chromosome starts at position 0, and given the coordinates *start-end* the *start* position is included in the interval and the end position is NOT included. So in a BED file, an interval that includes the first 5 bases of a chromosome would have start=0, end=5.

![1 vs 0 coordinates](img/annotation_coordSystems.png)

This might seem a little unintuitive, but is actually quite handy for doing certain calculations. For example, if we wanted to calculate the length of an interval, we just:

```
(interval end) - (interval start)
```

It is important to get comfortable with this concept, as other kinds of annotation files that we discuss later will use 1-based systems, and it is very easy to mix up the two and wind up with "off by 1" errors with your intervals, which can potentially matter quite a bit (e.g. transcription start sites or splicing junctions)!


Let's take a look at the example BED file we will be using today:

In [ ]:
%%bash
head data_day3/example.bed

We can see that in addition to the mandatory 3 columns, our BED file has an additional 4th column which contains information about the type of structural variant (SV) at that interval (insertion, deletion, inversion, etc.)

## Manipulating tabular files with Unix tools
Similar to day 1, we are going to start by covering some generic command line utilities that are included in most Unix-like systems which are helpful when working with tabular data, as well as revisiting some previously discussed commands. We are going to use BED files as an example, but the tools covered here apply to any sort of tabular data.

> A brief aside about *tab characters*:
> When you think of tab indentation, you might just imagine them as a bunch of spaces. However, in text parsing a tab is actually its own special *non-printing character*: `\t`. The `\t` character does not represent a letter or digit, but instead controls how text is displayed, i.e. it adds a horizontal tab indent.
> So technically, a line in a tab-separated file looks like this:
```
column1\tcolumn2\tcolumn3
```
> Which would then display on-screen like this:
```
column1 column2 column3
```

### `cut`
One of the most basic but useful commands when working with tables is the `cut` utility, which allows us to select and extract specific columns of a table. We use the `-f N` (`f`ield) to extract the Nth column of a tabular file.

In [ ]:
%%bash
cut -f 1 data_day3/example.bed | head

By default, `cut` uses the tab (`\t`) character as the column delimiter, but we can change the delimiter using the `-d '[delimiter]'` argument, e.g. a comma (`,`) instead of a tab:

In [ ]:
%%bash
cut -f 1 -d ',' data_day3/example.csv | head

If we want to select a *range* of columns, we use a `start column-end column` (minus symbol) with the `-f` argument (i.e. `-f start column-end column`); if we want to select *multiple specific* columns, we use `column A,column B` (comma) with `-f` (i.e. `-f A,B`)

In [ ]:
%%bash
#print columns 1 thru 3
cut -f 1-3 data_day3/example.bed | head

In [ ]:
%%bash
#print columns 1 and 4
cut -f 1,4 data_day3/example.bed | head

### `sort`
Another basic but crucial Unix tool is the `sort` function, which applies a sorting algorithm to reorder lines of a file.

In [ ]:
%%bash
sort data_day3/example.bed | head

By default, `sort` sorts based on the entire line *lexographically* (aka "dictionary order", attempts to sort alphabetical and numeric characters), but commonly we will want to sort based on *specific columns*. To do this, we use the `-k` (`k`ey column) to specify which column we want to use for sorting. 

However, there is a quirk with `sort`'s syntax that is important to understand. Say we want to sort on the 1st column of a BED file (i.e. sort by chromosome name). 

```
sort -k 1 example.bed
```

What this command actually says is "start sorting at the 1st column and *use the rest of the line*", which is not what we want! If we want to sort *exclusively* on a specific column, we need to explicitly *bound the range that `sort` looks at*; so, the proper syntax would be:

In [ ]:
%%bash
#Sort a bed file by the first column (chromosome name) 
sort -k 1,1 data_day3/example.bed | head

I.e. "sort by the first column and ONLY the first column". In our example, the sorted results look the same whether we explicitly set the range or not, but with other scenarios mixing up this syntax can lead to unexpected results!

#### Sorting number columns
As mentioned, `sort`'s algorithm sorts based on dictionary order. What happens when we try to sort a BED file on the numeric starting coordinate (i.e. the 2nd column)?

In [ ]:
%%bash
sort -k 2,2 data_day3/example.bed | head

We can see that trying to sort numbers using dictionary ordering does not work how we might expect it to where numbers are sorted lowest --> highest...for example, `10` and `100` are listed before `2`. To sort a column numerically, we need to use the `-n` argument: 

In [ ]:
%%bash
#sort by column 2 (and column 2 only), sorting numerically
sort -kn 2,2 data_day3/example.bed | head

We can see that now we are properly ordering numbers low --> high. 

If we wanted to sort in opposite order, high --> low, we could include another argument `-r` to `r`everse the sorting order:

In [ ]:
%%bash
sort -knr 2,2 data_day3/example.bed | head

> **Exercise**: write a command that extracts the annotations on `chr1`, and prints out the *final ten annotations* on the end of the chromosome, i.e. the ten annotations that occur last numerically. *Hint*: you will need **3** commands linked by **pipes** (one command we just looked at, the other two covered in prior workshops)...there are several ways you could do this!

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}

# get the chr1 lines with grep, sort by start position low -> high, get last 10 lines with tail
grep 'chr1' data_day3/example.bed | sort -kn 2,2 | tail

# OR

# get the chr1 lines with grep, REVERSE sort by start position high -> low, get first 10 lines with head
grep 'chr1' data_day3/example.bed | sort -knr 2,2 | head

#### Sorting multiple columns
We can also do some more complex sorting. By including multiple `-k` statements, we can *sequentially sort on multiple columns*. What this means is that `sort` will run multiple times on different specified columns, starting with the left-most listed column. So for example, the following command will sort first based on the start coordinate (column 2) THEN will sort based on the end coordinate (column 3):

In [ ]:
%%bash
sort -kn 2,2 -kn 3,3 data_day3/example.bed | head

> **Exercise**:
> Sort the BED file `data_day3/example.bed` by chromosome and start position, and save the output as a new file called `data_day3/example.sorted.bed`. 

In [ ]:
%%bash
## Command goes here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
sort -k 1,1 -kn 2,2 data_day3/example.bed > data_day3/example.sorted.bed

#### Getting unique values
The final use of `sort` that we will cover is the `-u` argument, which sorts a file and removes duplicate lines, i.e. retains only `u`nique lines! This can be useful on its own, but the real power of `sort -u` comes from combining it with other command lines tools, such as the previously covered `cut` command.

For example, the optional 4th column in our BED file contains information about the type of stuctural variant (SV) at that interval. If we wanted a unique list of all the different types of SVs that are in the file, we could use `cut` to grab the 4th column, then pipe it to `sort -u` to remove duplicates:

In [ ]:
%%bash
cut -f 4 data_day3/example.bed | sort -u

> **Exercise**:
> Write a command that *counts* how many chromosomes are in the file `data_day3/example.bed` (Hint: you will need 3 commands linked with pipes!)

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
cut -f 1 data_day3/example.bed | sort -u | wc -l

#### Building intuition: what's wrong with my BED file?
Trying to cut third column, returns a blank...why? 
Seperated by spaces, not tabs

Let's imagine you are working with a program that annotates repetitive elements in a genome. The program outputs a BED file (`data_day3/repeats.bed`) with the coordinates of the annotated repeats, with additional columns that contain information about the type of repetitive element (column 4) and what class of repetitive element family the repeat belongs to (column 5). You want a list of how many different repeat families are in your genome, so you run:

```
cut -f 5 data_day3/repeats.bed | sort -u | head
```

**Discussion**: what are we seeing instead of what we expected? What could we do to begin troubleshooting?

<details><summary>Solution</summary>

As always: first thing to do is look at the file!

```
less data_day3/repeats.bed
```

Nothing jumps out at me at first glance, so the next thing I would do is poke around and try different commands. What if I try looking at the 1st column instead?

```
cut -f 1 data_day3/repeats.bed | less
```

We can see in this case we DO see output to the screen...but it is the entire line, not just the first column? 

At this point I would begin to expect something is messed up with the formatting of the file. We didn't discuss this program yet, but `nano` is a light-weight text editor (like TextEdit or MS Word) that is available at the command line. It is a little beyond the scope of this class, so don't worry about the syntax, just know that `nano` lets me open and edit text files! When I open the BED file in `nano` and poke around:

```
nano data_day3/repeats.bed
```

We can see that the columns are NOT seperated by tab (`\t`) characters, but instead are separated by MULTIPLE WHITESPACES...which looks like a tab, but isn't! To deal with this, I would write a quick script or use some command line tricks (consult the internet or an LLM) to replace whitespaces with tabs, then I can re-run my `cut` command.

(This is actually an annoyingly common problem to run into, and is my own personal white whale...)
  
</details>

## `bedtools`
We can do a lot of simple processing of **BED** files with native bash commands like `grep`, `sort`, `wc`, etc. However, there are a lot of tasks that benefit from software built specifically for these types of files. For BED files (and other interval annotation files that we will cover later!), **bedtools** is a great tool. It has a wide range of functions for working with these files, allowing you to manipulate and transform BED files easily.

Like with `samtools` from last week, `bedtools` has a number of sub-commands/functions each with their own syntax. We'll only have time to go over a small number of `bedtools` functions in this workshop, so be sure to check out the [bedtools website](https://bedtools.readthedocs.io/en/latest/index.html) for more in-depth documentation on all its functions. Generally, the syntax will look like:

```
bedtools [sub-command] [arguments]
```

Replace the `[...]` with the relevant arguments or file paths, etc.

> **Motivation**: we're going to look at a BED file `data_day3/genome.bed`, which we can see contains an optional 4th column specifying the *type of genomic interval*, of which there are two: genes and exons. To demonstrate what `bedtools` can do, we will walk through the steps of adding intron annotations to our file. 

### `bedtools sort`
Before we do anything else, we need to do some tidying of our data. Much like with BAM/SAM files and `samtools`, other `bedtools` functions require a BED file to be *sorted* by chromosome and then by start position (i.e. column 1 then column 2), which we can do with `bedtools sort`:

In [ ]:
%%bash
#-i specifies the input file
#we direct the output to a new file with the .sorted.bed extension
bedtools sort -i data_day3/genome.bed > data_day3/genome.sorted.bed

> Note: you might remember that we did the exact same sorting earlier in the workshop with the generic command line tool `sort`! The `bedtools` manual even mentions this command, and that it is technically more efficient than using `bedtools sort`...but both tools do the same job, so you can use whichever you prefer!

For our desired task, we are going to need to separate out our `data_day3/genome.sorted.bed` BED file into seperate files based on their genomic location...in other words, we want seperate `gene.bed`, `exon.bed` and `cds.bed` files. How could we do this?

<details><summary>Solution</summary>

We can use our old friend `grep`! As we know the optional 4th column contains information about genomic location, we can just use seperate commands to make a file for each of our genomic features: 
  
</details>

In [ ]:
%%bash
#@title Solution {display-mode: "form"}

grep -i 'gene' data_day3/genome.sorted.bed > gene.bed
grep -i 'exon' data_day3/genome.sorted.bed > exon.bed

### Overlapping features with `bedtools subtract`
Given two interval files (**A** and **B**), `bedtools subtract` will find intervals in **B** that overlap an interval in **A**, *subtracts* the overlapping region and outputs the remaining region. If nothing overlaps a interval in **A**, the entire interval will be output.

![subtract](img/bedtools_subtract.png)

Syntax:
```
bedtools subtract -a [A.bed] -b [B.bed] > [output.bed]
```

So, if we have gene interval and exon interval annotations, subtracting the exons should give us the intronic intervals of each gene:

```
GENE:    [-----------------------------]
EXONS:        [+++++]     [++++]

INTRONS: [----]     [-----]    [-------]
```

In [ ]:
%%bash
bedtools subtract -a gene.bed -b exon.bed > intron.bed

> **Discussion**: let's not get too lost in the computational weeds, and re-center ourselves on biology. We want to get gene-level intron intervals. If we consider what we know about genes and transcripts and transcription, are there any potential issues with calculating introns this way?

<details><summary>Solution</summary>

This will work just fine for a single transcript, but we know that genes frequently have multiple transcripts, with differential intron chains, e.g.:

```
GENE:       [-----------------------------]
EXONS T1:        [+++++]     [++++]
EXONS T2:   [++++++]         [++++]

INTRONS T1: [----]     [-----]    [-------]
INTRONS T2:        [---------]    [-------]

```

We can see that with multiple splice variants, we will have overlapping, redundant intron annotations when we use only `bedtools subtract`. As we want annotations of the intronic regions for each gene, we would want to merge any overlapping intron intervals together...
  
</details>

### `bedtools merge`
![merge](img/bedtools_merge.png)

In a single BED file, `bedtools merge` takes any intervals that overlap or directly bookend each other and merges them together.

In [ ]:
%%bash
bedtools merge -i intron.bed > intron.merged.bed

> **Discussion**: using `merge` solves some of our issues, but let's take a step back. Are there any problems with calculating introns this way? What are some scenarios or questions it does not work for?

<details><summary>Solution</summary>

- It does not account for *strandedness* of the annotations
  - Transcripts can be on the `+` or `-` strand
  - Can account for with the `-s` option (tho our BED files don't have strand information...)
- Won't work if gene annotations contain un-translated region (3' or 5' UTRs)
  - `subtract` just checks what ISN'T exon in gene interval
- Most significantly: only looks at gene level and not individual transcripts
  - I.e. interval might be exonic or intronic in different splice variants, but `merge` will count it as "intron," e.g.:

```
GENE:       [-----------------------------]
EXONS T1:        [+++++]     [++++]
EXONS T2:   [++++++]         [++++]

INTRONS T1: [----]     [-----]    [-------]
INTRONS T2:        [---------]    [-------]

MERGED:     [----] [---------]    [-------]

``` 

In other words, defining intron this way gets us a gene-level annotation of introns, but ff we wanted intron intervals for each individual transcript (i.e. a transcript-level annotation), this approach would not be best and we'd likely need another tool.

</details>

### `bedtools intersect`
> Motivation: let's bring in additional info now. Say that we have a BED file with the coordinates of known structual variants (SVs) called `sv.bed`, and we want to see how many SVs occur within introns. In other words, we want to see how many intervals in `intron.bed` intersect with `sv.bed`.

![intersect](img/bedtools_intersect.png)

Syntax:
```
bedtools intersect -a A.bed -b B.bed
```

In [ ]:
%%bash
bedtools intersect -a intron.merged.bed -b sv.bed > intron_sv_overlap.bed

head intron_sv_overlap.bed

Given two interval files A and B, by default `bedtools intersect` will report just the shared interval between overlapping features in A and B. However, `intersect` has several arguments that can modify how intersections are reported:

- `-wa`: if interval in A overlaps interval in B, output entire A interval
- `-wb`: vice versa

In [ ]:
%%bash 
bedtools intersect -wa -a intron.merged.bed -b sv.bed | head

- `-loj`: "left outer join", reports every interval in A, and if it overlaps an interval in B report that overlap (otherwise "null")

In [ ]:
%%bash 
bedtools intersect -loj -a intron.merged.bed -b sv.bed | head

We can modify `bedtools intersect` in other ways as well. For example, the `-c` argument will report every interval in A, and add an additional column with a `c`ount of how many intervals in B overlap:

In [ ]:
%%bash 
bedtools intersect -c -a intron.merged.bed -b sv.bed | head

> **Exercise**: let's go back to our original question. We want to know how many introns contain *at least one structural variant*. Consult the `intersect` manual page to find the correct option and write a command to do this (Hint: you'll need a **pipe**...)

In [ ]:
%%bash
## Command here

In [ ]:
%%bash
#@title Solution {display-mode: "form"}
# -u outputs intervals in A that have at least one overlap in B
# we pipe that output to wc -l to count the number of lines
bedtools intersect -u -a intron.merged.bed -b sv.bed | wc -l

### `bedtools getfasta`
> Motivation: now we know how to identify which introns contain one or more structural variants. Let's say now that we want to look at the actual nucleotide sequence of those introns. Is there a way we can use a BED file to extract sequence from a FASTA file?

![getfasta](img/bedtools_getfasta.png)

If we have a BED file and a FASTA file whose header IDs EXACTLY match the chromosome IDs in the BED file, we can extract the sequence of each interval in the BED file and output it as a new FASTA file:

```
bedtools getfasta -fi [input.fasta] -bed [file.bed] -fo [output.fasta]
```
So, if we took the `bedtools intersect` command that we ran in the previous section and this time instead of piping it to `wc -l` we save the output to a BED file, we can then input that BED file to `getfasta` to extract the sequence of all introns that overlap an SV:

In [ ]:
%%bash
bedtools intersect -u -a intron.merged.bed -b sv.bed > introns_with_sv.bed

bedtools getfasta -fi genome.fa -bed introns_with_sv.bed > introns_with_sv.fa

### The many uses of `bedtools`
This is just a few of the most-commonly used functions of `bedtools`, but as we can see from the manual there are a TON more. Just to flash a few examples:

- Output all intervals NOT covered by an interval with `complement`
- Shift all intervals N bases with `shift`
- Calculate read coverage with `genomecov` or `coverage`
- Find the closest non-overlapping interval with `closest`

In other words, if you are ever trying to manipulate interval files, have a look thru the `bedtools` manual to see if the tool you are looking for already exists! 